# THE GATE — is "the model computes late" a raw-lens **artifact**?

`Runtime -> Run all`. For Qwen2.5 0.5B->7B, at each layer we compare:
- **raw lens** (the model's own head) — argmax = capital? (what we had: a late cliff)
- **probe** — a cross-validated linear probe trained to read the capital from that layer

If the **probe decodes the answer early** where the raw lens is flat, the early/mid layers DO
compute it (in a concept space the unembedding can't read) -> 'late' was an artifact, layers not
wasted. If even the probe is flat -> the model genuinely computes late. Prints the verdict + plot.

## 1 · env self-heal

In [ ]:
try:
    import numpy, scipy.sparse, matplotlib, sklearn  # noqa
    print('env OK — numpy', numpy.__version__)
except Exception as e:
    raise SystemExit('BROKEN ENV: '+repr(e)+'\n>>> Runtime -> Disconnect and delete runtime -> reconnect -> Run all.')


## 2 · code + deps

In [ ]:
import os
os.chdir('/content')
REPO='https://github.com/sinha-k-prat/small_fable.git'; BR='residual-orrery'
if not os.path.isdir('small_fable'):
    !git clone -q -b $BR $REPO
os.chdir('/content/small_fable')
!git fetch -q origin $BR && git reset --hard -q origin/$BR
!pip install -q bitsandbytes accelerate scikit-learn 'transformers>=4.44'
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')


## 3 · run the gate (drop 7B with `--models 0.5B 1.5B 3B` if GPU/disk is tight)

In [ ]:
!python tools/probe_lens.py --models 0.5B 1.5B 3B 7B --out probe
from IPython.display import Image, display
display(Image(filename='probe.png'))


## 4 · raw numbers

In [ ]:
import json, os
print(json.dumps(json.load(open('probe.json')), indent=2)[:2500])
try:
    from google.colab import files; files.download('probe.json'); files.download('probe.png')
except Exception: print('saved at', os.path.abspath('probe.json'))
